In [1]:
sample = 'mouse_skin'
save_dir = 'volcano'

In [ ]:
import cv2
import glob
import numpy as np
import pandas as pd
import scipy.io
from scipy.ndimage import rotate
from skimage import transform as tf
from skimage.transform import warp
import scipy.io
import cv2
import matplotlib.pyplot as plt
import os

all_cell_mapping = pd.read_csv(sample + '/all_cell_mapping.csv', index_col=0)
all_cell_raman = pd.read_csv(sample + '/all_cell_raman.csv', index_col=0)

In [ ]:
# Function to generate distinct colors from the 'jet' colormap
def generate_tab20_extended_colors():
    colors = list(plt.cm.tab20(np.linspace(0, 1, 20)))  # Generate 20 colors from tab20
    additional_color = plt.cm.Set2(5)  # Select a color from another colormap, e.g., the second color in 'Set1'
    colors.append(additional_color)  # Append the additional color to the list
    return colors

# Generate 21 colors
extended_colors = generate_tab20_extended_colors()
# define each cell type a color
color_mapping = {cell_type: color for cell_type, color in zip(all_cell_mapping['cell_type'].unique(), extended_colors)}
print(color_mapping)

In [ ]:
from scipy.integrate import simpson
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests
import plotly.graph_objs as go

wave_number = scipy.io.loadmat(sample + '/wavenumbers.mat')['wavenumber'][0][413:1286].round(0).astype(np.int16)

def normalize_spectra(wavenum, intensity, min_peak, max_peak):
    amide_mask = (wavenum >= min_peak) & (wavenum <= max_peak)
    area = simpson(intensity[amide_mask], x=wavenum[amide_mask])
    # print(area)
    return intensity / area


def calculate_DEP(data1, data2, wave_number, label_set, cell_type, prefix):

    import numpy as np
    import pandas as pd
    import scanpy as sc

    data1 = pd.DataFrame(data1, columns=wave_number)
    data2 = pd.DataFrame(data2, columns=wave_number)
    # Calculate the Wilcoxon rank-sum test for each feature (column)
    p_values = []
    log2FC = []
    for column in data1.columns:
        stat, p_value = ranksums(data1[column], data2[column])
        p_values.append(p_value)

        mean_data1 = data1[column].mean()
        mean_data2 = data2[column].mean()
        log2FC.append(np.log2(mean_data1 / mean_data2))
        
    # Apply FDR correction
    _, p_values_corrected, _, _ = multipletests(p_values, alpha=0.1, method='fdr_bh')

    # Create a DataFrame with the original and corrected p-values
    results = pd.DataFrame({
            'Index': range(len(data1.columns)),
            'Peak': data1.columns,
            'p-value': p_values,
            'p-value_corrected': p_values_corrected,
            'log2FC': log2FC
        })

    # Filter features with FDR < 0.05
    significant_results = results[results['p-value_corrected'] <= 0.1]

    upregulated = significant_results[significant_results['log2FC'] > 0]
    downregulated = significant_results[significant_results['log2FC'] < 0]

    upregulated = upregulated.sort_values('log2FC', ascending=False)
    downregulated = downregulated.sort_values('log2FC', ascending=True)
    
    if len(upregulated) < 1 and len(downregulated) < 1:
        return None
     
    # Print significant features
    # print('Number of DEFs:', len(significant_results), 'upregulated:', len(upregulated), 'downregulated:', len(downregulated))
    else:
        # plot_volcano

        print('number of upregulated peaks:', len(upregulated))
        print('number of downregulated peaks:', len(downregulated))

        # set figure size
        print('Top 50 upregulated peaks:')
        print(upregulated['Peak'].values[:50])
        print('Top 50 downregulated peaks:')
        print(downregulated['Peak'].values[:50])

        plt.figure(figsize=(8, 4))

        plt.scatter(x=results['log2FC'],y=results['p-value_corrected'].apply(lambda x:-np.log10(x)),s=2,label="Not significant", alpha=0.6, color="grey")

        # highlight down- or up- regulated genes
        down = results[(results['log2FC']<=0)&(results['p-value_corrected']<=0.05)]
        up = results[(results['log2FC']>=0)&(results['p-value_corrected']<=0.05)]

        plt.scatter(x=down['log2FC'],y=down['p-value_corrected'].apply(lambda x:-np.log10(x)),s=4,label="Decreased (" + str(len(down)) + ")",color="#91B9D6", alpha=0.8)
        plt.scatter(x=up['log2FC'],y=up['p-value_corrected'].apply(lambda x:-np.log10(x)),s=4,label="Increased (" + str(len(up)) + ")",color="#EB382E", alpha=0.8)

        # mark the top 3 up and down regulated genes consider log2FC 
        up = up.sort_values('log2FC',ascending=False)
        down = down.sort_values('log2FC',ascending=True)

        # for i,r in up.head(5).iterrows():
        #     plt.text(x=r['log2FC'],y=-np.log10(r['p-value_corrected']),s=up['Peak'][i], fontsize=8)

        # for i,r in down.head(5).iterrows():
        #     plt.text(x=r['log2FC'],y=-np.log10(r['p-value_corrected']),s=down['Peak'][i], fontsize=8)


        plt.xlabel("log2FC")
        plt.ylabel("-logFDR")

        # plt.axvline(-0.25,color="grey",linestyle="--")
        # plt.axvline(0.25,color="grey",linestyle="--")

        plt.axhline(-np.log10(0.05),color="grey",linestyle="--")
        plt.legend(loc='upper right', bbox_to_anchor=(1.38, 1))
        plt.title(f"{label_set} ({cell_type})")

        # show
                # save
        
        if '/' in cell_type:
            cell_type = cell_type.replace('/', '_')
            
        save_path = 'figures' + '/' + sample + '/' + save_dir
        if not os.path.exists(save_path):
            os.makedirs(save_path)


        # tight
        plt.tight_layout()

        plt.savefig(f'{save_path}/{prefix}_{label_set}_{cell_type}.pdf')
       
        plt.show()
        

In [ ]:
from scipy import stats

label_sets = ['old']
tiltle_sets = ['old']


for label_set,tiletle_set in zip(label_sets, tiltle_sets):

    selected_cell_mapping = all_cell_mapping[all_cell_mapping['sample_type'] == 'O']

    if len(selected_cell_mapping) < 1:
        continue

    selected_cell_raman = all_cell_raman.loc[selected_cell_mapping.index]

    labels = selected_cell_mapping['p21+'].values * 1
    features = selected_cell_raman.values.astype(np.float32)
    
    new_features = []
    for i in range(features.shape[0]):
        new_features.append(normalize_spectra(wave_number, features[i], 1630, 1700))
    features = np.stack(new_features, axis=0)
    
    positive = features[labels == 1]
    negative = features[labels == 0]
    
    if len(positive) < 1 and len(negative) < 1:
        continue

    calculate_DEP(positive, negative, wave_number, label_set, 'global', 'SvsNS')


# cell_type
for label_set,tiletle_set in zip(label_sets, tiltle_sets):

    selected_cell_mapping = all_cell_mapping[all_cell_mapping['sample_type'] == 'O'] # select only old cells
    cell_types = set(selected_cell_mapping['cell_type'])

    for cell_type in cell_types: 

        selected_cell_type_mapping = selected_cell_mapping[selected_cell_mapping['cell_type'] == cell_type]

        if len(selected_cell_mapping) < 1:
            continue

        selected_cell_raman = all_cell_raman.loc[selected_cell_type_mapping.index]

        labels = selected_cell_type_mapping['p21+'].values * 1
        features = selected_cell_raman.values.astype(np.float32)
        
        new_features = []
        for i in range(features.shape[0]):
            new_features.append(normalize_spectra(wave_number, features[i], 1630, 1700))
        features = np.stack(new_features, axis=0)
        
        positive = features[labels == 1]
        negative = features[labels == 0]
        
        if len(positive) < 1 and len(negative) < 1:
            continue

        calculate_DEP(positive, negative, wave_number, label_set, cell_type, 'SvsNS')
